# MM-CTR Challenge — Kaggle


In [1]:
%%time
import os, time, zipfile, datetime, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(N_GPUS):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
print(f'Device  : {DEVICE}  ({N_GPUS} GPU(s))')
if DEVICE.type == 'cpu':
    raise RuntimeError('No GPU — enable: Settings -> Accelerator -> GPU T4 x2')


PyTorch : 2.10.0+cu128
CUDA    : True
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB
Device  : cuda  (2 GPU(s))
CPU times: user 2.88 s, sys: 906 ms, total: 3.79 s
Wall time: 5.79 s


In [2]:
DATASET_SLUG   = 'microlens-data'
INPUT_DIR      = f'/kaggle/input/datasets/mehdiouassif/{DATASET_SLUG}/'
DATA_DIR       = INPUT_DIR + 'data/MicroLens_1M_x1/'
ITEM_EMB_PATH  = INPUT_DIR + 'data/item_emb.parquet'
ITEM_INFO_PATH = INPUT_DIR + 'data/item_info.parquet'
WORKING_DIR    = '/kaggle/working/'
SUBMISSION_DIR = WORKING_DIR + 'submission/'
CHECKPOINT_DIR = WORKING_DIR + 'checkpoints/'
FEATURES_DIR   = WORKING_DIR + 'features/'
for d in [SUBMISSION_DIR, CHECKPOINT_DIR, FEATURES_DIR]:
    os.makedirs(d, exist_ok=True)

ITEM_EMB_CACHE = FEATURES_DIR + 'item_emb_final_v9.npy'
CKPT_PATH      = CHECKPOINT_DIR + 'sasrec_dcnv2_best_v9.pt'

for p, name in [(DATA_DIR,'DATA_DIR'),(ITEM_EMB_PATH,'ITEM_EMB_PATH'),(ITEM_INFO_PATH,'ITEM_INFO_PATH')]:
    status = 'OK  ' if os.path.exists(p) else 'MISS'
    print(f'  {status} {name}: {p}')

MAX_SEQ_LEN = 64
SEED        = 42
torch.manual_seed(SEED); np.random.seed(SEED); torch.cuda.manual_seed_all(SEED)

TAG_EMB_DIM    = 64
ITEM_EMB_DIM   = 128
MAX_TAGS       = 64
T1_BLEND_ALPHA = 0.4

MAX_EPOCHS = 40
WARMUP_EPS = 3
CFG = dict(
    max_epochs   = MAX_EPOCHS,
    warmup_eps   = WARMUP_EPS,
    cosine_t_max = MAX_EPOCHS - WARMUP_EPS,
    max_lr       = 1e-3,
    min_lr       = 1e-5,
    patience     = 8,
    train_bs     = 4096 * max(N_GPUS, 1),
    valid_bs     = 4096 * max(N_GPUS, 1),
    label_smooth = 0.02,
)
print('Config ready')
print(f'  train_bs={CFG["train_bs"]}  max_lr={CFG["max_lr"]}  patience={CFG["patience"]}')


  OK   DATA_DIR: /kaggle/input/datasets/mehdiouassif/microlens-data/data/MicroLens_1M_x1/
  OK   ITEM_EMB_PATH: /kaggle/input/datasets/mehdiouassif/microlens-data/data/item_emb.parquet
  OK   ITEM_INFO_PATH: /kaggle/input/datasets/mehdiouassif/microlens-data/data/item_info.parquet
Config ready
  train_bs=8192  max_lr=0.001  patience=8


In [3]:
%%time
print('Loading data...')
df_train = pd.read_parquet(DATA_DIR + 'train.parquet')
df_valid = pd.read_parquet(DATA_DIR + 'valid.parquet')
df_test  = pd.read_parquet(DATA_DIR + 'test.parquet')
df_items = pd.read_parquet(ITEM_INFO_PATH)
df_emb   = pd.read_parquet(ITEM_EMB_PATH)

LABEL_COL = next((c for c in ['label','click','Click','Label'] if c in df_train.columns), None)
assert LABEL_COL, f'No label column in {list(df_train.columns)}'
SEQ_COL = next((c for c in ['item_seq','item_sequence','his_item_id'] if c in df_train.columns), None)
ID_COL  = next((c for c in ['ID','id','sample_id'] if c in df_test.columns), df_test.columns[0])

all_items = pd.concat([df_train['item_id'], df_valid['item_id'],
                        df_test['item_id'],  df_items['item_id']])
N_ITEMS         = int(all_items.max())
ITEM_VOCAB_SIZE = N_ITEMS + 1
del all_items

all_users = pd.concat([df_train['user_id'], df_valid['user_id'], df_test['user_id']])
USER_VOCAB_SIZE = int(all_users.max()) + 2
del all_users

LIKES_VOCAB_SIZE = (int(df_train['likes_level'].max()) + 2) if 'likes_level' in df_train.columns else 2
VIEWS_VOCAB_SIZE = (int(df_train['views_level'].max()) + 2) if 'views_level' in df_train.columns else 2

tag_col_detect = next((c for c in ['item_tags','tag_id','tags','tag_ids','tag']
                        if c in df_items.columns), None)
if tag_col_detect:
    all_tags = [int(t) for tags in df_items[tag_col_detect]
                for t in (tags if isinstance(tags,(list,np.ndarray)) else []) if int(t) != 0]
    TAG_VOCAB_SIZE = max(all_tags) + 1 if all_tags else 2
else:
    TAG_VOCAB_SIZE = 2

pretrained_col = next((c for c in ['item_emb_d128_v3','item_emb_d128_v2','item_emb_d128']
                        if c in df_emb.columns), None)
if pretrained_col:
    _sample = np.asarray(df_emb.iloc[0][pretrained_col])
    ITEM_EMB_DIM = int(_sample.shape[0])

if os.path.exists(ITEM_EMB_CACHE):
    _cached = np.load(ITEM_EMB_CACHE)
    if _cached.shape != (N_ITEMS, ITEM_EMB_DIM):
        os.remove(ITEM_EMB_CACHE)
        print(f'Deleted stale cache {_cached.shape} != expected ({N_ITEMS},{ITEM_EMB_DIM})')
    del _cached

ctr        = df_train[LABEL_COL].mean()
POS_WEIGHT = (1.0 - ctr) / ctr

print(f'Label:{LABEL_COL}  Seq:{SEQ_COL}  ID:{ID_COL}')
print(f'Train:{len(df_train):,}  Valid:{len(df_valid):,}  Test:{len(df_test):,}')
print(f'N_ITEMS:{N_ITEMS:,}  USER_VOCAB:{USER_VOCAB_SIZE:,}')
print(f'TAG_VOCAB:{TAG_VOCAB_SIZE:,}  LIKES_VOCAB:{LIKES_VOCAB_SIZE}  VIEWS_VOCAB:{VIEWS_VOCAB_SIZE}')
print(f'ITEM_EMB_DIM:{ITEM_EMB_DIM}  pretrained_col:{pretrained_col}')
print(f'CTR={ctr:.4f}  pos_weight={POS_WEIGHT:.2f}')


Loading data...
Label:label  Seq:item_seq  ID:ID
Train:3,600,000  Valid:10,000  Test:379,142
N_ITEMS:91,717  USER_VOCAB:1,000,002
TAG_VOCAB:11,740  LIKES_VOCAB:12  VIEWS_VOCAB:12
ITEM_EMB_DIM:128  pretrained_col:item_emb_d128_v3
CTR=0.2500  pos_weight=3.00
CPU times: user 10.8 s, sys: 9.88 s, total: 20.7 s
Wall time: 13.5 s


In [4]:
%%time

class TagEncoder(nn.Module):
    def __init__(self, vocab_size, tag_dim=TAG_EMB_DIM, out_dim=ITEM_EMB_DIM, max_tags=MAX_TAGS):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, tag_dim, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(tag_dim, 256), nn.LayerNorm(256), nn.ReLU(),
            nn.Dropout(0.1), nn.Linear(256, out_dim),
        )
    def forward(self, tag_ids):
        mask   = (tag_ids != 0).float().unsqueeze(-1)
        x      = self.emb(tag_ids) * mask
        pooled = x.sum(1) / mask.sum(1).clamp(min=1)
        return self.mlp(pooled)


def infonce_loss(anchors, positives, temperature=0.07):
    a = F.normalize(anchors,   dim=-1)
    p = F.normalize(positives, dim=-1)
    logits = torch.mm(a, p.T) / temperature
    labels = torch.arange(len(a), device=a.device)
    return F.cross_entropy(logits, labels)


if os.path.exists(ITEM_EMB_CACHE):
    print(f'Loading cached embeddings: {ITEM_EMB_CACHE}')
    final_emb = np.load(ITEM_EMB_CACHE)
    print(f'  Shape: {final_emb.shape}')
else:
    print('Building tag + pretrained tensors...')
    tag_col = tag_col_detect
    print(f'  tag_col={tag_col}  pretrained_col={pretrained_col}  dim={ITEM_EMB_DIM}')

    tags_tensor       = torch.zeros(ITEM_VOCAB_SIZE, MAX_TAGS,     dtype=torch.long)
    pretrained_tensor = torch.zeros(ITEM_VOCAB_SIZE, ITEM_EMB_DIM, dtype=torch.float32)
    emb_id_map = dict(zip(df_emb['item_id'].values, range(len(df_emb))))

    for _, row in tqdm(df_items.iterrows(), total=len(df_items), desc='Tensors'):
        iid = int(row['item_id'])
        if not (1 <= iid <= N_ITEMS): continue
        if tag_col is not None:
            raw = row[tag_col]
            if isinstance(raw, (list, np.ndarray)):
                t = [int(x) for x in raw if int(x) != 0][:MAX_TAGS]
            elif isinstance(raw, str) and raw.strip():
                t = [int(x) for x in raw.replace(' ','').split(',') if x][:MAX_TAGS]
            else: t = []
            pads = t + [0]*(MAX_TAGS - len(t))
            tags_tensor[iid] = torch.tensor(pads, dtype=torch.long)
        if pretrained_col and iid in emb_id_map:
            arr = np.asarray(df_emb.iloc[emb_id_map[iid]][pretrained_col], dtype=np.float32)
            if arr.shape[0] == ITEM_EMB_DIM:
                pretrained_tensor[iid] = torch.tensor(arr)

    tags_tensor       = tags_tensor.to(DEVICE)
    pretrained_tensor = pretrained_tensor.to(DEVICE)

    print('Building co-occurrence pairs...')
    pairs = set()
    for seq in tqdm(df_train[SEQ_COL].values, desc='Pairs'):
        if not isinstance(seq, (list, np.ndarray)): continue
        ids = [int(x) for x in seq if int(x) != 0]
        for i in range(len(ids)-1):
            a, b = ids[i], ids[i+1]
            if 1 <= a <= N_ITEMS and 1 <= b <= N_ITEMS:
                pairs.add((a, b))
    pairs = list(pairs)
    has_pairs = len(pairs) > 0
    print(f'  Pairs: {len(pairs):,}')

    _tag_model = TagEncoder(vocab_size=TAG_VOCAB_SIZE + 1).to(DEVICE)
    tag_opt    = Adam(_tag_model.parameters(), lr=1e-3)
    all_ids    = torch.arange(1, N_ITEMS+1)
    id_loader  = DataLoader(TensorDataset(all_ids), batch_size=4096*max(N_GPUS,1), shuffle=True)
    if has_pairs:
        pair_loader = DataLoader(
            TensorDataset(torch.tensor(pairs, dtype=torch.long)),
            batch_size=2048*max(N_GPUS,1), shuffle=True)

    T1_EPOCHS = 30
    print(f'Training TagEncoder {T1_EPOCHS} epochs...')
    t0_te = time.time()
    for ep in range(1, T1_EPOCHS+1):
        _tag_model.train()
        total, nb_b = 0.0, 0
        pair_iter = iter(pair_loader) if has_pairs else None
        for (item_ids,) in id_loader:
            item_ids = item_ids.to(DEVICE)
            out  = _tag_model(tags_tensor[item_ids])
            mse  = F.mse_loss(out, pretrained_tensor[item_ids])
            if pair_iter is not None:
                try:    (pb,) = next(pair_iter)
                except: pair_iter = iter(pair_loader); (pb,) = next(pair_iter)
                pb  = pb.to(DEVICE)
                nce = infonce_loss(_tag_model(tags_tensor[pb[:,0]]),
                                   _tag_model(tags_tensor[pb[:,1]]))
            else: nce = torch.tensor(0.0, device=DEVICE)
            loss = 0.5*mse + 0.5*nce
            tag_opt.zero_grad(); loss.backward(); tag_opt.step()
            total += loss.item(); nb_b += 1
        if ep == 1:
            elapsed = time.time()-t0_te
            print(f'  Epoch 1 in {elapsed:.1f}s — est: {elapsed*T1_EPOCHS/60:.1f} min')
        if ep % 5 == 0 or ep == 1:
            print(f'  Ep {ep:3d}/{T1_EPOCHS} | Loss={total/nb_b:.4f}')

    _tag_model.eval()
    with torch.no_grad():
        _ids    = torch.arange(1, N_ITEMS+1, device=DEVICE)
        tag_out = _tag_model(tags_tensor[_ids]).cpu().numpy()
        pre_np  = pretrained_tensor[_ids].cpu().numpy()

    zero_mask = np.abs(pre_np).sum(axis=1) == 0
    final_emb = np.where(
        zero_mask[:,None],
        tag_out,
        T1_BLEND_ALPHA * tag_out + (1-T1_BLEND_ALPHA) * pre_np
    ).astype(np.float32)
    np.save(ITEM_EMB_CACHE, final_emb)
    print(f'Saved: {ITEM_EMB_CACHE}  shape={final_emb.shape}')
    del tags_tensor, pretrained_tensor, _tag_model
    torch.cuda.empty_cache()

assert final_emb.shape == (N_ITEMS, ITEM_EMB_DIM), \
    f'Shape mismatch: got {final_emb.shape}, expected ({N_ITEMS},{ITEM_EMB_DIM})'
print(f'Item embeddings ready: {final_emb.shape} OK')


Building tag + pretrained tensors...
  tag_col=item_tags  pretrained_col=item_emb_d128_v3  dim=128


Tensors:   0%|          | 0/91718 [00:00<?, ?it/s]

Building co-occurrence pairs...


Pairs:   0%|          | 0/3600000 [00:00<?, ?it/s]

  Pairs: 5,834,674
Training TagEncoder 30 epochs...
  Epoch 1 in 3.4s — est: 1.7 min
  Ep   1/30 | Loss=4.5392
  Ep   5/30 | Loss=4.2638
  Ep  10/30 | Loss=4.2176
  Ep  15/30 | Loss=4.1918
  Ep  20/30 | Loss=4.1671
  Ep  25/30 | Loss=4.1515
  Ep  30/30 | Loss=4.1364
Saved: /kaggle/working/features/item_emb_final_v9.npy  shape=(91717, 128)
Item embeddings ready: (91717, 128) OK
CPU times: user 2min 28s, sys: 6.33 s, total: 2min 34s
Wall time: 2min 33s


In [5]:
class Dice(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.alpha = nn.Parameter(torch.zeros(dim))
        self.bn    = nn.BatchNorm1d(dim, eps=1e-8)
    def forward(self, x):
        p = torch.sigmoid(self.bn(x))
        return p*x + (1.0-p)*self.alpha*x


class CrossLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim)
    def forward(self, x0, x):
        return x0 * self.linear(x) + x


class SASRecDCNv2CTR(nn.Module):
    ITEM_ID_DIM = 64
    MM_DIM      = 128
    SEQ_DIM     = 192   # ITEM_ID_DIM + MM_DIM
    USER_DIM    = 64
    LIKES_DIM   = 16
    VIEWS_DIM   = 16
    DCN_IN      = 672   # 384+192+64+16+16
    DEEP_OUT    = 256
    COMBINED    = 928   # DCN_IN + DEEP_OUT

    def __init__(self, final_emb, item_vocab, user_vocab, likes_vocab, views_vocab):
        super().__init__()
        self.item_id_emb = nn.Embedding(item_vocab, self.ITEM_ID_DIM, padding_idx=0)
        mm_pad = np.zeros((item_vocab, self.MM_DIM), dtype=np.float32)
        # safe: slice final_emb in case ITEM_EMB_DIM > MM_DIM
        mm_pad[1:len(final_emb)+1] = final_emb[:, :self.MM_DIM]
        self.mm_emb = nn.Embedding.from_pretrained(
            torch.tensor(mm_pad), freeze=True, padding_idx=0)
        self.user_emb  = nn.Embedding(user_vocab,  self.USER_DIM)
        # FIX: use inferred vocab sizes, not hardcoded 12
        self.likes_emb = nn.Embedding(likes_vocab, self.LIKES_DIM)
        self.views_emb = nn.Embedding(views_vocab, self.VIEWS_DIM)
        # FIX: pos_enc size = MAX_SEQ_LEN+1, dim = SEQ_DIM
        # positions are 0-indexed (0..L-1), so size just needs to be >= MAX_SEQ_LEN
        self.pos_enc = nn.Embedding(MAX_SEQ_LEN + 1, self.SEQ_DIM)
        nn.init.normal_(self.pos_enc.weight, 0, 0.1)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=self.SEQ_DIM, nhead=4, dim_feedforward=768,
            dropout=0.2, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=4)
        self.cross = nn.ModuleList([CrossLayer(self.DCN_IN) for _ in range(3)])
        self.deep  = nn.Sequential(
            nn.Linear(self.DCN_IN, 1024), Dice(1024), nn.Dropout(0.2),
            nn.Linear(1024, 512),         Dice(512),  nn.Dropout(0.2),
            nn.Linear(512, self.DEEP_OUT),Dice(self.DEEP_OUT), nn.Dropout(0.2),
        )
        # FIX: Dice in pred head (was ReLU — kills gradients on negative activations)
        self.pred = nn.Sequential(
            nn.Linear(self.COMBINED, 256), Dice(256), nn.Dropout(0.2),
            nn.Linear(256, 128),           Dice(128), nn.Dropout(0.1),
            nn.Linear(128, 1),
        )
        self._init_weights()
        self.freeze_item_id()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding) and m.padding_idx is not None:
                nn.init.normal_(m.weight, 0, 0.01)
                m.weight.data[m.padding_idx].zero_()

    def freeze_item_id(self):   self.item_id_emb.weight.requires_grad_(False)
    def unfreeze_item_id(self): self.item_id_emb.weight.requires_grad_(True)

    def get_item_repr(self, ids):
        return torch.cat([self.item_id_emb(ids), self.mm_emb(ids)], dim=-1)

    def forward(self, item_seq, target_item, user_id, likes, views):
        B, L = item_seq.shape
        seq_r = self.get_item_repr(item_seq)                                  # (B,L,192)
        pos   = torch.arange(L, device=item_seq.device).unsqueeze(0)         # 0-indexed
        seq_r = seq_r + self.pos_enc(pos)
        pad_m = (item_seq == 0)
        seq_out = self.transformer(seq_r, src_key_padding_mask=pad_m)         # (B,L,192)
        lengths  = (item_seq != 0).sum(1).clamp(min=1)
        last_idx = (lengths - 1).clamp(min=0)
        last_out = seq_out[torch.arange(B, device=item_seq.device), last_idx] # (B,192)
        target_repr = self.get_item_repr(target_item)                         # (B,192)
        scores  = torch.bmm(target_repr.unsqueeze(1),
                             seq_out.transpose(1,2)).squeeze(1) / (self.SEQ_DIM**0.5)
        scores  = scores.masked_fill(pad_m, float('-inf'))
        weights = torch.softmax(scores, dim=1)
        attended = torch.bmm(weights.unsqueeze(1), seq_out).squeeze(1)        # (B,192)
        seq_repr = torch.cat([last_out, attended], dim=-1)                   # (B,384)
        x = torch.cat([seq_repr, target_repr,
                        self.user_emb(user_id),
                        self.likes_emb(likes),
                        self.views_emb(views)], dim=-1)                       # (B,672)
        x0 = xc = x
        for layer in self.cross: xc = layer(x0, xc)
        xd  = self.deep(x)
        return self.pred(torch.cat([xc, xd], dim=-1)).squeeze(-1)            # raw logits


# Sanity-check forward pass before any training
_tmp = SASRecDCNv2CTR(final_emb,
    item_vocab=ITEM_VOCAB_SIZE, user_vocab=USER_VOCAB_SIZE,
    likes_vocab=LIKES_VOCAB_SIZE, views_vocab=VIEWS_VOCAB_SIZE)
_tmp.eval()
with torch.no_grad():
    _o = _tmp(torch.zeros(2,MAX_SEQ_LEN,dtype=torch.long),
              torch.ones(2,dtype=torch.long), torch.ones(2,dtype=torch.long),
              torch.zeros(2,dtype=torch.long), torch.zeros(2,dtype=torch.long))
assert _o.shape == (2,), f'Bad output shape: {_o.shape}'
total_p   = sum(p.numel() for p in _tmp.parameters())
trainable = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f'Model OK: {total_p:,} params  ({trainable:,} trainable, item_id frozen)')
print(f'Forward shape: {_o.shape} ✓')
del _tmp


Model OK: 86,381,729 params  (68,771,873 trainable, item_id frozen)
Forward shape: torch.Size([2]) ✓


In [6]:
%%time

def parse_seq(seq, max_len=MAX_SEQ_LEN):
    if isinstance(seq, (list, np.ndarray)):
        ids = [int(x) for x in seq if int(x) != 0]
    elif isinstance(seq, str) and seq.strip():
        ids = [int(x) for x in seq.replace(' ','').split(',') if x and x != '0']
    else: ids = []
    return ([0]*(max_len - len(ids[-max_len:])) + ids[-max_len:])


class CTRDataset(Dataset):
    def __init__(self, df, has_label=True):
        print(f'  Parsing {len(df):,} rows...')
        seqs = [parse_seq(s) for s in tqdm(df[SEQ_COL], leave=False)]
        self.item_seq = torch.tensor(seqs, dtype=torch.long)
        self.item_id  = torch.tensor(df['item_id'].values, dtype=torch.long)
        self.user_id  = torch.tensor(df['user_id'].values, dtype=torch.long)
        # FIX: clamp to inferred vocab size (not hardcoded 11)
        max_lv_likes = LIKES_VOCAB_SIZE - 2
        max_lv_views = VIEWS_VOCAB_SIZE - 2
        self.likes = (torch.tensor(df['likes_level'].clip(0,max_lv_likes).values.astype(int),
                       dtype=torch.long) if 'likes_level' in df.columns
                      else torch.zeros(len(df), dtype=torch.long))
        self.views = (torch.tensor(df['views_level'].clip(0,max_lv_views).values.astype(int),
                       dtype=torch.long) if 'views_level' in df.columns
                      else torch.zeros(len(df), dtype=torch.long))
        self.has_label = has_label
        if has_label:
            self.labels = torch.tensor(df[LABEL_COL].values, dtype=torch.float32)

    def __len__(self): return len(self.item_id)
    def __getitem__(self, idx):
        base = (self.item_seq[idx], self.item_id[idx], self.user_id[idx],
                self.likes[idx], self.views[idx])
        return base + (self.labels[idx],) if self.has_label else base


print('Building datasets...')
train_ds = CTRDataset(df_train)
valid_ds = CTRDataset(df_valid)
test_ds  = CTRDataset(df_test, has_label=False)

NW = 2
train_loader = DataLoader(train_ds, batch_size=CFG['train_bs'], shuffle=True,
                          num_workers=NW, pin_memory=True, persistent_workers=True)
valid_loader = DataLoader(valid_ds, batch_size=CFG['valid_bs'], shuffle=False,
                          num_workers=NW, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['valid_bs'], shuffle=False,
                          num_workers=NW, pin_memory=True, persistent_workers=True)
print(f'Train:{len(train_loader)} Valid:{len(valid_loader)} Test:{len(test_loader)} batches')


Building datasets...
  Parsing 3,600,000 rows...


  0%|          | 0/3600000 [00:00<?, ?it/s]

  Parsing 10,000 rows...


  0%|          | 0/10000 [00:00<?, ?it/s]

  Parsing 379,142 rows...


  0%|          | 0/379142 [00:00<?, ?it/s]

Train:440 Valid:2 Test:47 batches
CPU times: user 1min 10s, sys: 2.93 s, total: 1min 13s
Wall time: 1min 12s


In [7]:
%%time
_raw_model = SASRecDCNv2CTR(
    final_emb,
    item_vocab=ITEM_VOCAB_SIZE, user_vocab=USER_VOCAB_SIZE,
    likes_vocab=LIKES_VOCAB_SIZE, views_vocab=VIEWS_VOCAB_SIZE,
).to(DEVICE)
model = nn.DataParallel(_raw_model) if N_GPUS > 1 else _raw_model

pw        = torch.tensor([POS_WEIGHT], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
optimizer = Adam(_raw_model.parameters(), lr=5e-4, weight_decay=1e-5)
cosine_sched = CosineAnnealingLR(optimizer, T_max=CFG['cosine_t_max'], eta_min=CFG['min_lr'])
scaler       = torch.cuda.amp.GradScaler()
best_auc, no_imp = 0.0, 0

used  = torch.cuda.memory_allocated(0)/1e6
total_vram = torch.cuda.get_device_properties(0).total_memory/1e6
print(f'VRAM: {used:.0f}/{total_vram:.0f} MB')
print(f'Epochs={CFG["max_epochs"]}  Warmup={CFG["warmup_eps"]}  Patience={CFG["patience"]}')
print('-'*65)

for epoch in range(1, CFG['max_epochs']+1):

    # ── LR schedule ───────────────────────────────────────────────────────────
    if epoch <= CFG['warmup_eps']:
        lr = 5e-4 * epoch / CFG['warmup_eps']
        for pg in optimizer.param_groups: pg['lr'] = lr
    else:
        cosine_sched.step()
        lr = optimizer.param_groups[0]['lr']

    # ── Unfreeze item_id_emb after warmup ─────────────────────────────────────
    if epoch == CFG['warmup_eps'] + 1:
        _raw_model.unfreeze_item_id()  # always on _raw_model
        # FIX: restart LR to max_lr and reset cosine scheduler properly
        # (re-creating CosineAnnealingLR mid-loop was resetting step counter wrong)
        for pg in optimizer.param_groups: pg['lr'] = CFG['max_lr']
        cosine_sched.base_lrs = [CFG['max_lr']] * len(cosine_sched.base_lrs)
        cosine_sched.last_epoch = 0
        lr = CFG['max_lr']
        print(f'  item_id_emb unfrozen | LR restarted to {lr:.2e}')

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    epoch_loss, t0 = 0.0, time.time()
    for batch in tqdm(train_loader, desc=f'Ep {epoch:02d} train', leave=False):
        seqs, iids, uids, lk, vw, labels = [b.to(DEVICE) for b in batch]
        ls = CFG['label_smooth']
        labels_s = labels * (1-ls) + 0.5*ls
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(seqs, iids, uids, lk, vw)
            loss   = criterion(logits, labels_s)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        # FIX: clip on _raw_model.parameters() not model (DP/compile wrapper)
        nn.utils.clip_grad_norm_(_raw_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    ep_time = time.time()-t0
    if epoch == 1:
        print(f'  Epoch 1: {ep_time:.1f}s — est total: {ep_time*CFG["max_epochs"]/60:.1f} min')

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc=f'Ep {epoch:02d} valid', leave=False):
            seqs, iids, uids, lk, vw, labels = [b.to(DEVICE) for b in batch]
            with torch.cuda.amp.autocast():
                logits = model(seqs, iids, uids, lk, vw)
            all_p.extend(torch.sigmoid(logits).cpu().numpy())
            all_l.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_l, all_p)
    print(f'Ep {epoch:02d}/{CFG["max_epochs"]} | Loss={epoch_loss/len(train_loader):.4f} '
          f'| AUC={auc:.4f} | LR={lr:.2e} | {ep_time:.0f}s', end='')

    if auc > best_auc:
        best_auc = auc; no_imp = 0
        # FIX: save _raw_model.state_dict() — model wrapper may change between runs
        torch.save({'epoch':epoch,'auc':auc,'model_state':_raw_model.state_dict()}, CKPT_PATH)
        print(' <- best')
    else:
        no_imp += 1; print()
        if no_imp >= CFG['patience']:
            print(f'  Early stop (best AUC={best_auc:.4f})')
            break

print('-'*65)
print(f'Best validation AUC: {best_auc:.4f}')
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
_raw_model.load_state_dict(ckpt['model_state'])
print(f'Best weights restored (epoch {ckpt["epoch"]})')


VRAM: 380/15637 MB
Epochs=40  Warmup=3  Patience=8
-----------------------------------------------------------------


Ep 01 train:   0%|          | 0/440 [00:00<?, ?it/s]

  Epoch 1: 327.0s — est total: 218.0 min


Ep 01 valid:   0%|          | 0/2 [00:00<?, ?it/s]

Ep 01/40 | Loss=1.0811 | AUC=0.5133 | LR=1.67e-04 | 327s <- best


...

Ep 12 valid:   0%|          | 0/2 [00:00<?, ?it/s]

Ep 12/40 | Loss=0.1358 | AUC=0.8738 | LR=8.90e-04 | 332s


Ep 13 train:   0%|          | 0/440 [00:00<?, ?it/s]

Ep 13 valid:   0%|          | 0/2 [00:00<?, ?it/s]

Ep 13/40 | Loss=0.1324 | AUC=0.8733 | LR=8.62e-04 | 333s
  Early stop (best AUC=0.9214)
-----------------------------------------------------------------
Best validation AUC: 0.9214
Best weights restored (epoch 5)
CPU times: user 1h 12min 52s, sys: 2min 12s, total: 1h 15min 4s
Wall time: 1h 12min 12s


In [8]:
%%time
print(f'Inference on {len(df_test):,} test rows...')
_raw_model.eval()
scores = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Inference'):
        seqs, iids, uids, lk, vw = [b.to(DEVICE) for b in batch]
        with torch.cuda.amp.autocast():
            logits = _raw_model(seqs, iids, uids, lk, vw)
        scores.extend(torch.sigmoid(logits).cpu().numpy().tolist())

scores = np.array(scores, dtype=np.float32)
print(f'Score range  : [{scores.min():.4f}, {scores.max():.4f}]')
print(f'Score mean   : {scores.mean():.4f}  (expected ~{ctr:.4f})')
print(f'Score median : {np.median(scores):.4f}')
assert len(scores) == len(df_test), f'Row mismatch: {len(scores)} vs {len(df_test)}'
print('Row count OK')


Inference on 379,142 test rows...


Inference:   0%|          | 0/47 [00:00<?, ?it/s]

Score range  : [0.0056, 1.0000]
Score mean   : 0.6223  (expected ~0.2500)
Score median : 0.8506
Row count OK
CPU times: user 19.1 s, sys: 332 ms, total: 19.4 s
Wall time: 20.5 s


In [9]:
import datetime, zipfile, os  # safe re-import

exp_id   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = '/kaggle/working/prediction.csv'
zip_path = f'/kaggle/working/submission_{exp_id}.zip'

# FIX: competition expects Task1, Task2, Task1&2 columns — not a single 'label'
sub = pd.DataFrame({
    ID_COL:      df_test[ID_COL].values,
    'Task1':     scores,
    'Task2':     scores,
    'Task1&2':   scores,
})
sub.to_csv(csv_path, index=False)

# FIX: file inside zip must be named 'prediction.csv'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname='prediction.csv')

print(f'Rows    : {len(sub):,}')
print(f'Columns : {list(sub.columns)}')
print(f'CSV     : {csv_path}')
print(f'ZIP     : {zip_path}  (contains prediction.csv)')
print()
print(sub.head(5).to_string(index=False))


Rows    : 379,142
Columns : ['ID', 'Task1', 'Task2', 'Task1&2']
CSV     : /kaggle/working/prediction.csv
ZIP     : /kaggle/working/submission_20260506_212451.zip  (contains prediction.csv)

 ID    Task1    Task2  Task1&2
  0 0.724121 0.724121 0.724121
  1 0.993652 0.993652 0.993652
  2 0.863281 0.863281 0.863281
  3 0.018539 0.018539 0.018539
  4 0.026611 0.026611 0.026611


In [10]:
print('='*60)
print('FINAL RESULTS')
print('='*60)
print(f'Best Val AUC  : {best_auc:.4f}')
print(f'Target        : 0.9000')
print(f'Delta         : {best_auc-0.90:+.4f}')
print()

FINAL RESULTS
Best Val AUC  : 0.9214
Target        : 0.9000
Delta         : +0.0214

